# Demo: from SMILES + sequence to a KIBA affinity prediction

A short walk-through of the `dti_gt` package: featurise a molecule, encode a protein, run the full model, and inspect a finished training run. Run from the `phase1/` directory with the project virtual environment (`python -m ipykernel install --user --name dti_gt` if the kernel is missing). Outputs are intentionally stripped.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent  # notebook started from notebooks/
sys.path.insert(0, str(ROOT / 'src'))

import torch
import pandas as pd
from dti_gt.utils.seed import set_seed
set_seed(0)
print('root:', ROOT)

## 1. SMILES → molecular graph

In [ ]:
from dti_gt.data.featurize import smiles_to_graph, ATOM_FEATURE_DIMS, BOND_FEATURE_DIMS

imatinib = 'Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1'
g = smiles_to_graph(imatinib, drug_id='imatinib')
print(g)
print('atom feature vocab sizes:', ATOM_FEATURE_DIMS)
print('bond feature vocab sizes:', BOND_FEATURE_DIMS)
print('first atom (categorical codes):', g.x[0].tolist(), ' degree:', int(g.degree[0]))

## 2. Protein sequence → features (ProtBERT-BFD embedding shipped for the 229 KIBA kinases)

In [ ]:
from dti_gt.data.proteins import load_protein_embeddings, sequence_to_tokens, format_for_protbert

index_by_target, emb = load_protein_embeddings(ROOT / 'data/protein_embeddings/kiba_prot_bert_bfd.npz')
print('embeddings:', tuple(emb.shape), '| first target:', next(iter(index_by_target)))

targets = pd.read_csv(ROOT / 'data/kiba/targets.csv')
seq = targets.iloc[0]['sequence']
print('sequence length', len(seq), '| ProtBERT input looks like:', format_for_protbert(seq)[:40], '...')
print('token encoding (cnn variant):', sequence_to_tokens(seq, max_len=1000)[:12])

## 3. Build the full model from a config and run a forward pass on a mini-batch

In [ ]:
from torch_geometric.data import Batch
from dti_gt.utils.config import load_config
from dti_gt.models.dti_model import build_model, count_parameters

cfg = load_config(ROOT / 'configs/default.yaml')
model = build_model(cfg, protein_input_dim=emb.shape[1]).eval()
print(f'{count_parameters(model):,} trainable parameters')

drugs = pd.read_csv(ROOT / 'data/kiba/drugs.csv').head(4)
graphs = [smiles_to_graph(s, str(d)) for d, s in zip(drugs['drug_id'], drugs['smiles'])]
batch = Batch.from_data_list(graphs)
prot = emb[:4].clone()
with torch.no_grad():
    z = model(batch, prot)  # z-scored KIBA units for an untrained model
print('raw outputs:', z.squeeze().tolist())

## 4. Inspect a finished run (after `python scripts/train.py --config configs/default.yaml`)

In [ ]:
import matplotlib.pyplot as plt

runs = sorted((ROOT / 'results/runs').glob('*/metrics.json'))
print(len(runs), 'runs found')
if runs:
    run_dir = runs[0].parent
    m = json.loads(runs[0].read_text())
    print(run_dir.name, '| test:', {k: round(v, 4) for k, v in m['test'].items()})
    h = pd.read_csv(run_dir / 'history.csv')
    ax = h.plot(x='epoch', y=['train_mse', 'val_mse'], figsize=(6, 3.5), grid=True, title=run_dir.name)
    ax.set_ylabel('MSE (KIBA units)')
    plt.show()